In [19]:
import pandas as pd
import requests
import ipaddress
from datetime import datetime
import concurrent.futures

# API Keys
VT_API_KEY = "a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08"
OTX_API_KEY = "ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe"

# Headers for API requests
VT_HEADERS = {"x-apikey": VT_API_KEY}
OTX_HEADERS = {"X-OTX-API-KEY": OTX_API_KEY}

# API URLs
OTX_URL_IP = "https://otx.alienvault.io/api/v1/indicators/IPv4/{}/general"
OTX_URL_DOMAIN = "https://otx.alienvault.io/api/v1/indicators/domain/{}/general"
OTX_URL_HOSTNAME = "https://otx.alienvault.io/api/v1/indicators/hostname/{}"

def is_ip(value):
    """ Check if the given value is a valid IP address. """
    try:
        ipaddress.ip_address(value)
        return True
    except ValueError:
        return False

def determine_query_type(query):
    """ Determine if the query is an IP, domain, or hostname. """
    if is_ip(query):
        return "ip"
    elif "." in query:
        return "hostname"
    else:
        return "domain"

def fetch_virustotal_data(query):
    """Fetch data from VirusTotal API for IP or Domain using ThreatConnect enrich endpoint."""
    import urllib.parse

    # indicator_values should be a list of indicator values to enrich
    # For compatibility, if query is a single value, wrap in list
    indicator_values = [query] if isinstance(query, str) else query
    enriched_results = []

    for value in indicator_values:
        try:
            # Use the indicator *value*, not the ID
            encoded_value = urllib.parse.quote(value)
            enrich_url = f'/v3/indicators/{encoded_value}/enrich?type=Shodan&type=VirusTotalV3'
            ro.set_http_method('POST')
            ro.set_request_uri(enrich_url)
            ro.set_body({})
            enrich_response = tc.api_request(ro)

            if enrich_response.status_code == 200:
                enrich_data = enrich_response.json()
                enrich_data['summary'] = value  # Save the value as key
                enriched_results.append(enrich_data)
        except Exception as e:
            continue
    # Unnest the 'data' field in enriched_results if it's a list of dicts
    if enriched_results:
        # If enriched_results is a list of dicts with 'data' key, flatten each
        flattened = []
        for item in enriched_results:
            if isinstance(item, dict) and 'data' in item:
                flat = item.copy()
                flat.update(pd.json_normalize(item['data']).iloc[0].to_dict() if isinstance(item['data'], dict) else {})
                del flat['data']
                flattened.append(flat)
            else:
                flattened.append(item)
        enriched_results = flattened

    # Return first result if only one value, else list
    if len(enriched_results) == 1:
        return enriched_results[0]
    
    return enriched_results

def fetch_otx_data(query):
    """Fetch data from OTX API for IP, Domain, or Hostname."""
    query_type = determine_query_type(query)

    if query_type == "ip":
        url = OTX_URL_IP.format(query)
    elif query_type == "hostname":
        url = OTX_URL_HOSTNAME.format(query)
    else:
        url = OTX_URL_DOMAIN.format(query)

    try:
        response = requests.get(url, headers=OTX_HEADERS, verify=False)
        response.raise_for_status()
        data = response.json()

        return {
            "search_term": query,
            "base_indicator": data.get('base_indicator', {}),
            "reputation": data.get('reputation'),
            "geo": data.get('geo', {}),
            "whois": data.get('whois', {}),
            "open_ports": data.get('open_ports', []),
            "link": f"https://otx.alienvault.com/indicator/{query_type}/{query}"
        }

    except Exception as e:
        print(f"OTX Error for {query}: {e}")
        return None

def unnest_base_indicator(df):
    """ Unnest the 'base_indicator' column and create separate columns for its keys. """
    if 'base_indicator' not in df.columns:
        return df

    base_df = pd.json_normalize(df['base_indicator'])
    base_df.columns = [f"base_{col}" for col in base_df.columns]

    df = df.drop(columns=['base_indicator']).reset_index(drop=True)
    df = pd.concat([df, base_df], axis=1)
    return df

def process_indicator(indicator, observed_by, partner_count):
    """Fetch data for a single indicator."""
    timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

    with concurrent.futures.ThreadPoolExecutor() as executor:
        vt_future = executor.submit(fetch_virustotal_data, indicator)
        otx_future = executor.submit(fetch_otx_data, indicator)

        vt_data = vt_future.result()
        otx_data = otx_future.result()

    if vt_data:
        vt_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    if otx_data:
        otx_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    return vt_data, otx_data

def main(recent_tags):
    """ Main function to process indicators. """
    if 'summary' not in recent_tags.columns:
        print("The 'summary' column is missing.")
        return pd.DataFrame(), pd.DataFrame()

    search_terms = recent_tags['summary'].dropna().unique().tolist()
    print(f"Processing {len(search_terms)} unique search terms.")

    vt_results = []
    otx_results = []

    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = []
        for indicator in search_terms:
            partners = recent_tags.loc[recent_tags['summary'] == indicator, 'partners'].values
            partner_count = recent_tags.loc[recent_tags['summary'] == indicator, 'partner_count'].values
            observed_by = partners[0] if len(partners) > 0 else "N/A"
            partner_count = partner_count[0] if len(partner_count) > 0 else "N/A"

            futures.append(executor.submit(process_indicator, indicator, observed_by, partner_count))

        for future in concurrent.futures.as_completed(futures):
            vt_data, otx_data = future.result()
            if vt_data:
                vt_results.append(vt_data)
            if otx_data:
                otx_results.append(otx_data)

    vt_df = pd.DataFrame(vt_results)
    otx_df = pd.DataFrame(otx_results)

    
    otx_df = unnest_base_indicator(otx_df)

    return vt_df, otx_df

if __name__ == "__main__":
    filtered_recent_tags = pd.read_excel(
        r'Z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Spreadsheet\I&W_indicators_full_20251008.xlsx'
    )
    vt_df, otx_df = main(filtered_recent_tags)
    print("Script completed successfully.")

Processing 4 unique search terms.


C:\Users\jaskew\AppData\Local\Temp\ipykernel_23092\1756684870.py:126: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


Script completed successfully.


In [ ]:
import os
import pandas as pd
import docx
from datetime import datetime
from docx import Document
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# File paths
TEMPLATE_PATH = r"z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\I&W Report Template.docx"
#TEMPLATE_PATH = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\I&W Report Template.docx"
OUTPUT_DIR = r"z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Generated Reports"
#OUTPUT_DIR = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated Reports"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

def consolidate_sources(vt_df, otx_df):
    """ Consolidate links from both DataFrames for each search term. """
    # Handle None or empty DataFrames
    if vt_df is None or vt_df.empty:
        vt_links = pd.DataFrame(columns=['search_term', 'vt_links'])
    else:
        vt_links = vt_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
        vt_links.columns = ['search_term', 'vt_links']

    if otx_df is None or otx_df.empty:
        otx_links = pd.DataFrame(columns=['search_term', 'otx_links'])
    else:
        otx_links = otx_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
        otx_links.columns = ['search_term', 'otx_links']

    # Merge the two link sets
    if not vt_links.empty and not otx_links.empty:
        consolidated = pd.merge(vt_links, otx_links, on='search_term', how='outer')
    elif not vt_links.empty:
        consolidated = vt_links.copy()
        consolidated['otx_links'] = ''
    elif not otx_links.empty:
        consolidated = otx_links.copy()
        consolidated['vt_links'] = ''
    else:
        return pd.DataFrame(columns=['search_term', 'sources'])

    # Combine the links, handling NaN values
    consolidated['sources'] = consolidated[['vt_links', 'otx_links']].fillna('').apply(
        lambda x: ', '.join(filter(None, x)), axis=1
    )

    return consolidated[['search_term', 'sources']]

def extract_date(timestamp):
    """ Extract only the date from the timestamp. """
    if pd.isna(timestamp) or timestamp == 'N/A':
        return 'N/A'
    
    # Handle datetime object or string
    try:
        # Attempt to parse as a datetime object
        if isinstance(timestamp, str):
            timestamp = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
        return timestamp.strftime("%Y-%m-%d")
    except Exception:
        return 'N/A'

def populate_table(table, data):
    """ Populate a Word table with the given data. """
    # Iterate over data and populate rows
    for index, row in data.iterrows():
        # Insert a new row before the last row (template row)
        new_row = table.add_row().cells
        new_row[0].text = str(row.get('search_term', 'N/A'))
        new_row[1].text = str(row.get('type', 'N/A'))
        new_row[2].text = extract_date(row.get('observed_date', 'N/A'))
        # For the 'observed_by_otx' column, stack values instead of comma-separating
        # observed_by_otx: stack values (one per line) from OTX and VirusTotal "observed_by" columns if present
        observed_by_otx = ''
        observed_by_list = []

        # Try to get observed_by from OTX
        if 'observed_by_otx' in row and pd.notna(row['observed_by_otx']):
            observed_by_list.extend([v.strip() for v in str(row['observed_by_otx']).split(',') if v.strip()])
        elif 'observed_by' in row and pd.notna(row['observed_by']):
            observed_by_list.extend([v.strip() for v in str(row['observed_by']).split(',') if v.strip()])

        # Remove duplicates and join with newlines
        if observed_by_list:
            observed_by_otx = '\n'.join(sorted(set(observed_by_list)))
        else:
            observed_by_otx = 'N/A'
        new_row[3].text = str(observed_by_otx)
        new_row[4].text = str(row.get('notes', ''))
        
def fill_word_template(template_path, output_path, df, group_id):
    """ Fill the template with data and place sources outside the table. """
    if not os.path.exists(template_path):
        print(f"Template not found: {template_path}")
        return
    
    try:
        doc = Document(template_path)

        # Populate the table
        table = None
        for tbl in doc.tables:
            if "Indicators/Identifiers" in tbl.rows[0].cells[0].text:
                table = tbl
                break

        if table:
            populate_table(table, df)

        # Find and replace placeholders outside the table
        for para in doc.paragraphs:
            if "{{ipaddress}}" in para.text:
                # Try to get the first search_term as the IP address
                ip_address = str(df['search_term'].iloc[0]) if 'search_term' in df.columns else 'N/A'
                para.text = para.text.replace("{{ipaddress}}", ip_address)
            if "{{asn}}" in para.text:
                # Try to get ASN from vt_df if available, else use 'N/A'
                asn_value = str(df['asn'].iloc[0]) if 'asn' in df.columns and not df['asn'].isna().all() else 'N/A'
                para.text = para.text.replace("{{asn}}", asn_value)
            if "{{whois}}" in para.text:
                # Try to get WHOIS info from otx_df if available, else use 'N/A'
                whois_value = str(df['whois'].iloc[0]) if 'whois' in df.columns and not df['whois'].isna().all() else 'N/A'
                para.text = para.text.replace("{{whois}}", whois_value)
            if "{{partners}}" in para.text:
                # Try to get partners from recent_tags if available, else use 'N/A'
                partners_value = ''
                if 'search_term' in df.columns and not df.empty:
                    search_term = df['search_term'].iloc[0]
                    partners_row = recent_tags[recent_tags['summary'] == search_term]
                    if not partners_row.empty and 'partners' in partners_row.columns:
                        partners_value = str(partners_row['partners'].iloc[0])
                if not partners_value:
                    partners_value = 'N/A'
                para.text = para.text.replace("{{partners}}", partners_value)
            if "{{weblink}}" in para.text:
                weblink_value = ''
                # Try to get the first search_term as the indicator
                weblink_value = ''
                if 'search_term' in df.columns and not df.empty:
                    search_term = df['search_term'].iloc[0]
                    # Try to find a 'webLink' in df for the indicator (if present)
                    if 'webLink' in df.columns:
                        match = df[df['search_term'] == search_term]
                        if not match.empty and pd.notna(match['webLink'].iloc[0]):
                            weblink_value = str(match['webLink'].iloc[0])
                    # Fallback: try 'link' column if 'webLink' is not present or empty
                    if not weblink_value and 'link' in df.columns:
                        match = df[df['search_term'] == search_term]
                        if not match.empty and pd.notna(match['link'].iloc[0]):
                            weblink_value = str(match['link'].iloc[0])
                para.text = para.text.replace("{{weblink}}", "")
                if weblink_value:
                    # Add hyperlink using WordprocessingML (only show the link, no display text)
                    r_id = doc.part.relate_to(
                        weblink_value, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True
                    )
                    hyperlink = OxmlElement('w:hyperlink')
                    hyperlink.set(qn('r:id'), r_id)
                    new_run = OxmlElement('w:r')
                    rPr = OxmlElement('w:rPr')
                    rStyle = OxmlElement('w:rStyle')
                    rStyle.set(qn('w:val'), 'Hyperlink')
                    rPr.append(rStyle)
                    new_run.append(rPr)
                    t = OxmlElement('w:t')
                    t.text = weblink_value
                    new_run.append(t)
                    hyperlink.append(new_run)
                    para._p.append(hyperlink)
                else:
                    para.text = "N/A"
            if "{{sources}}" in para.text:
                # Build sources as hyperlinks if possible

                # Helper to add a hyperlink to a paragraph
                def add_hyperlink(paragraph, url):
                    # Create the w:hyperlink tag and add needed values
                    part = paragraph.part
                    r_id = part.relate_to(url, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True)
                    hyperlink = OxmlElement('w:hyperlink')
                    hyperlink.set(qn('r:id'), r_id)
                    # Create a w:r element
                    new_run = OxmlElement('w:r')
                    # Create a w:rPr element
                    rPr = OxmlElement('w:rPr')
                    # Add color and underline for hyperlink style
                    rStyle = OxmlElement('w:rStyle')
                    rStyle.set(qn('w:val'), 'Hyperlink')
                    rPr.append(rStyle)
                    new_run.append(rPr)
                    # Create a w:t element and set the text
                    t = OxmlElement('w:t')
                    t.text = url
                    new_run.append(t)
                    hyperlink.append(new_run)
                    paragraph._p.append(hyperlink)

                # Remove the placeholder
                para.text = para.text.replace("{{sources}}", "")

                # Add each source as a hyperlink, stacked (one per line, no commas)
                sources = []
                for srcs in df['sources'].dropna().unique():
                    for src in [s.strip() for s in srcs.split(',') if s.strip()]:
                        sources.append(src)
                for i, src in enumerate(sources):
                    add_hyperlink(para, src)
                    if i < len(sources) - 1:
                        para.add_run().add_break()  # Add line break instead of comma

        # --- Save the document ---
        current_date = datetime.now().strftime("%Y-%m-%d")
        folder_path = os.path.join(OUTPUT_DIR, current_date)
        os.makedirs(folder_path, exist_ok=True)

        # Use indicator name for single indicators, group_id for multiple indicators
        if len(df) == 1:
            # Single indicator - use indicator name
            indicator_name = str(df['search_term'].iloc[0])
            sanitized_name = indicator_name.replace(":", "_").replace("/", "_").replace(" ", "_")
            output_path = os.path.join(folder_path, f"I&W_Report_{sanitized_name}.docx")
        else:
            # Multiple indicators - use group_id
            sanitized_group_id = str(group_id).replace(":", "_").replace("/", "_").replace(" ", "_")
            output_path = os.path.join(folder_path, f"I&W_Report_Group_{sanitized_group_id}.docx")
        doc.save(output_path)
        print(f"Saved report: {output_path}")

    except Exception as e:
        print(f"Error while generating report for group {group_id}: {e}")

def main(vt_df, otx_df, recent_tags):
    # Normalize VT columns you rely on
    # Only rename columns if vt_df is not empty and not None
    if vt_df is not None and not vt_df.empty:
        vt_df = vt_df.rename(columns={'ip': 'search_term', 'webLink': 'link'})

    # Normalize recent_tags to 'search_term' so we can join on one key
    if isinstance(recent_tags, pd.DataFrame) and not recent_tags.empty:
        recent_norm = recent_tags.rename(columns={'summary': 'search_term'}).copy()
        keep_cols = [c for c in ['search_term','observations','type','partners','observed_date','webLink','group_id'] if c in recent_norm.columns]
        recent_norm = recent_norm[keep_cols]
    else:
        recent_norm = pd.DataFrame(columns=['search_term'])

    # Combine VT and OTX
    if vt_df is None or vt_df.empty:
        combined_df = otx_df.copy()
    else:
        combined_df = pd.merge(vt_df, otx_df, on='search_term', how='outer', suffixes=('_vt', '_otx'))

    # Consolidate sources
    sources_df = consolidate_sources(vt_df, otx_df)
    combined_df = pd.merge(combined_df, sources_df, on='search_term', how='left')

    # Merge recent tags (already on 'search_term') with suffixes to preserve type
    combined_df = pd.merge(combined_df, recent_norm, on='search_term', how='left', suffixes=('', '_tag'))
    
    # Ensure we use the indicator type from recent_tags if available
    if 'type_tag' in combined_df.columns:
        combined_df['type'] = combined_df['type_tag']
        combined_df.drop(columns=['type_tag'], inplace=True)

    # Group by group_id and generate one report per group
    if 'group_id' in combined_df.columns:
        print(f"Grouping indicators by group_id...")
        
        # Get unique group_ids, excluding null/NaN values
        unique_groups = combined_df['group_id'].dropna().unique()
        
        print(f"Found {len(unique_groups)} unique groups to process")
        
        for group_id in unique_groups:
            # Get all indicators for this group
            group_df = combined_df[combined_df['group_id'] == group_id]
            
            if not group_df.empty:
                print(f"Processing group {group_id} with {len(group_df)} indicators")
                fill_word_template(TEMPLATE_PATH, None, group_df, group_id)
            else:
                print(f"No indicators found for group {group_id}")
        
        # Handle indicators without group_id (create individual reports)
        ungrouped_df = combined_df[combined_df['group_id'].isna()]
        if not ungrouped_df.empty:
            print(f"Processing {len(ungrouped_df)} ungrouped indicators...")
            for indicator in ungrouped_df['search_term'].dropna().unique():
                indicator_df = ungrouped_df[ungrouped_df['search_term'] == indicator]
                sanitized = str(indicator).replace(":", "_").replace("/", "_")
                fill_word_template(TEMPLATE_PATH, None, indicator_df, sanitized)
    else:
        print("No group_id column found, generating individual reports...")
        # Fallback to original behavior if no group_id column
        for indicator in combined_df['search_term'].dropna().unique():
            indicator_df = combined_df[combined_df['search_term'] == indicator]
            sanitized = str(indicator).replace(":", "_").replace("/", "_")
            fill_word_template(TEMPLATE_PATH, None, indicator_df, sanitized)

    
if __name__ == "__main__":
    # If vt_df is empty or None, use otx_df and recent_tags only
    if vt_df is None or vt_df.empty:
        main(None, otx_df, filtered_recent_tags)
    else:
        main(vt_df, otx_df, filtered_recent_tags)

Grouping indicators by group_id...
Found 1 unique groups to process
Processing group 5629499544001057.0 with 3 indicators
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated Reports\2025-10-08\I&W_Report_Group_5629499544001057.0.docx
Processing 1 ungrouped indicators...
Saved report: C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated Reports\2025-10-08\I&W_Report_104.21.20.66.docx
